In [49]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
dataset = pd.read_csv("C:\\Users\\Acer\\Downloads\\ORDER_DATA_LAT_LON.csv")
vehicle_data = pd.read_csv("C:\\Users\\Acer\\Downloads\\VEHICLE_DATA_LAT_LON.csv")


In [50]:
dataset.head()

,job_id,date,unloading_time_min,start_time,end_time,lat,long,vol_m3,weight_kg
0,205921,02-01-2024,16,10:00,23:00,51.541782,-0.119571,3.64,82
1,205922,02-01-2024,22,10:00,23:00,51.549544,-0.317543,1.77,103
2,216810,02-01-2024,17,10:00,23:00,51.542744,-0.044280,3.02,128
3,216811,02-01-2024,22,10:00,23:00,51.607627,-0.480161,2.44,61
4,223881,02-01-2024,11,10:00,23:00,51.401156,-0.000819,2.85,79


In [51]:
# dataset2 = dataset.reset_index()52.506885,-1.728302
loc_data = dataset[['lat','long']]
depot = pd.DataFrame({'lat': [52.506885], 'long': [-1.728302]})
loc_data_with_depot = pd.concat([depot, loc_data], ignore_index=True)   
loc_data_with_depot = loc_data_with_depot.reset_index()
loc_data_with_depot.head()

,index,lat,long
0,0,52.506885,-1.728302
1,1,51.541782,-0.119571
2,2,51.549544,-0.317543
3,3,51.542744,-0.044280
4,4,51.607627,-0.480161


In [52]:
vehicle_data.head()

,Type,Name,Max Volume C3 m,Max Weight Kg,Max working Hours,Fixed Cost,Per Km Cost
0,1,Bolero_Max,16,2000,11.0,3800,20
1,2,Bolero,12,1500,10.0,3200,20
2,3,Pickup,10,1200,9.5,2500,18
3,4,Chhota_Hathi,8,1000,9.0,2100,15
4,5,Bolero_Max,16,2000,11.0,3800,20


In [53]:

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in km
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    
    distance_km = R * c
    return int(distance_km)

# Convert distance to driving time at 30 km/h
def calculate_driving_time(distance_km):
    speed_kmh = 30  # Average speed in km/h
    time_hours = distance_km / speed_kmh
    return int(time_hours * 3600)  # Convert hours to seconds


In [54]:
import itertools
locations = loc_data_with_depot.set_index("index")[["lat", "long"]].to_dict("index")

# Initialize matrices as dictionaries
dist_matrix = {}
time_matrix = {}

# Compute distance and time matrices
for (idx1, coord1), (idx2, coord2) in itertools.combinations(locations.items(), 2):
    dist = haversine(coord1["lat"], coord1["long"], coord2["lat"], coord2["long"])
    time = calculate_driving_time(dist)
    
    dist_matrix[(idx1, idx2)] = dist
    dist_matrix[(idx2, idx1)] = dist  # Symmetric
    time_matrix[(idx1, idx2)] = time
    time_matrix[(idx2, idx1)] = time  # Symmetric

# Include self-distance as 0
for idx in locations.keys():
    dist_matrix[(idx, idx)] = 0.0
    time_matrix[(idx, idx)] = 0

# Print matrices
print("Distance Matrix:", dist_matrix)
print("\nTime Matrix:", time_matrix)

Distance Matrix: {(0, 1): 153, (1, 0): 153, (0, 2): 143, (2, 0): 143, (0, 3): 157, (3, 0): 157, (0, 4): 131, (4, 0): 131, (0, 5): 170, (5, 0): 170, (0, 6): 148, (6, 0): 148, (0, 7): 177, (7, 0): 177, (0, 8): 147, (8, 0): 147, (0, 9): 149, (9, 0): 149, (0, 10): 158, (10, 0): 158, (0, 11): 144, (11, 0): 144, (0, 12): 176, (12, 0): 176, (0, 13): 152, (13, 0): 152, (0, 14): 159, (14, 0): 159, (0, 15): 150, (15, 0): 150, (0, 16): 155, (16, 0): 155, (0, 17): 150, (17, 0): 150, (0, 18): 167, (18, 0): 167, (0, 19): 155, (19, 0): 155, (0, 20): 140, (20, 0): 140, (0, 21): 151, (21, 0): 151, (0, 22): 160, (22, 0): 160, (0, 23): 154, (23, 0): 154, (0, 24): 146, (24, 0): 146, (0, 25): 165, (25, 0): 165, (0, 26): 165, (26, 0): 165, (0, 27): 167, (27, 0): 167, (0, 28): 158, (28, 0): 158, (0, 29): 141, (29, 0): 141, (0, 30): 146, (30, 0): 146, (0, 31): 152, (31, 0): 152, (0, 32): 155, (32, 0): 155, (0, 33): 168, (33, 0): 168, (0, 34): 161, (34, 0): 161, (0, 35): 140, (35, 0): 140, (0, 36): 160, (36, 0

In [55]:
nodes = list(loc_data_with_depot['index'])
print(nodes)
vehicles = [i for i in range(len(vehicle_data))]
print(vehicles)
demands_w=[0]+dataset["weight_kg"].tolist()
Q1 = vehicle_data["Max Weight Kg"].tolist()
max_capacity_w = {i: Q1[i] for i in range(len(vehicles))}
var_cost = vehicle_data["Per Km Cost"].tolist()
fixed_cost = vehicle_data["Fixed Cost"].tolist()
start_time = [600] + (pd.to_datetime(dataset["start_time"],
                                             format='%H:%M').dt.hour * 60 + pd.to_datetime(
    dataset["start_time"],format='%H:%M').dt.minute).tolist()
print(start_time)
finish_time = [1380] + (pd.to_datetime(dataset["end_time"],
                                             format='%H:%M').dt.hour * 60 + pd.to_datetime(
    dataset["end_time"],format='%H:%M').dt.minute).tolist()
service_time = [0] + dataset["unloading_time_min"].tolist()
print(service_time)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
[600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600]
[0, 16, 22, 17, 22, 11, 10, 33, 29, 10, 10, 14, 10, 12, 21, 24, 10, 10, 14, 10, 10, 10, 14, 12, 14, 15, 18, 41, 10, 28, 21, 22, 20, 35, 37, 30, 27, 12]


In [56]:
import copy
import time
import numpy as np
import itertools
import cProfile

# Start measuring time
starting_time = time.time()

def is_valid_capacity(route, demands_w, max_capacity_w):
    total_weight = 0
    for node in route:
        if node != 0:  # Exclude depot
            total_weight += demands_w[node]
            if total_weight > int(max_capacity_w):
                return False
    return True

def is_valid_time_window(route, time_matrix, start_time, finish_time,service_time):
    current_time = 0  # Start at time 0
    for i in range(len(route) - 1):  # Traverse the route using indices
        node = route[i]
        next_node = route[i + 1]

        # Update the current time to simulate travel and waiting
        current_time += time_matrix[node, next_node]
        if current_time < start_time[next_node]:  # Wait for time window
            current_time = start_time[next_node]

        # Check if we are within the time window
        if current_time + service_time > finish_time[next_node]:
            return False
    return True

def is_valid_route(route, demands_w, max_capacity_w, time_matrix, start_time, finish_time,service_time):
    if not is_valid_capacity(route, demands_w, max_capacity_w):
        return False

    for i in range(len(route) - 1):
        next_node = route[i + 1]
        if (finish_time[next_node] - start_time[next_node]) < 1560:
            if not is_valid_time_window(route, time_matrix, start_time, finish_time,service_time):
                return False

    return True


def initialize_solution(nodes, vehicles, dist_matrix, demands_w, max_capacity_w, time_matrix, start_time, finish_time):
    s_t = time.time()
    solution = {v: [] for v in vehicles}
    remaining_demand_w = copy.deepcopy(demands_w)
    unvisited = set(nodes[1:])  # Exclude depot

    for v in vehicles:
        current_node = 0  # Start at the depot
        current_capacity_w = max_capacity_w[v]
        current_time = 0  # Start at time 0
        route = [current_node]

        while unvisited:
            # Find the nearest feasible node
            nearest_node = None
            nearest_distance = float("inf")
            for n in unvisited:
                # Check capacity and time window feasibility
                if remaining_demand_w[n] <= current_capacity_w and dist_matrix[current_node, n] < nearest_distance:
                    # Simulate arrival time
                    arrival_time = current_time + time_matrix[current_node, n]
                    if arrival_time <= finish_time[n]:
                        nearest_node = n
                        nearest_distance = dist_matrix[current_node, n]

            if nearest_node is None:
                break  # No valid node found for this vehicle, end its route

            # Add the nearest node to the route
            route.append(nearest_node)
            current_capacity_w -= remaining_demand_w[nearest_node]
            remaining_demand_w[nearest_node] = 0
            unvisited.remove(nearest_node)
            current_time += time_matrix[current_node, nearest_node]
            if current_time < start_time[nearest_node]:  # Wait for the time window to open
                current_time = start_time[nearest_node]
            current_node = nearest_node

        route.append(0)  # Return to depot
        solution[v] = route

        # Stop if all customers have been visited
        if not unvisited:
            break

    
    e_t = time.time()
    run_time = e_t - s_t
    print(f"Time for initial solution: {run_time:.2f} seconds")
    return solution

In [57]:
def generate_neighbors(solution, vehicles, nodes, tabu_list, max_capacity_w, demands_w, dist_matrix):
    """
    Generate neighbors for the given solution using relocation, swap, 2-opt moves, 
    and merging routes of smaller vehicles into a larger vehicle.
    Optimized for reduced runtime.
    """
    s_t = time.time()
    neighbors = []

    # Cache current weights for each vehicle
    route_weights = {
        v: sum(demands_w[node] for node in solution[v] if node != 0) for v in vehicles
    }
    
    # Relocation: Move a customer from one vehicle to another
    for v1, v2 in itertools.permutations(vehicles, 2):
        for i in range(1, len(solution[v1]) - 1):  # Exclude depot
            node = solution[v1][i]
            for j in range(1, len(solution[v2])):  # Allow insertions in v2
                # Modify routes incrementally
                route_v1 = solution[v1][:]
                route_v2 = solution[v2][:]
                route_v1.remove(node)
                route_v2.insert(j, node)

                # Incremental weight checks
                new_weight_v1 = route_weights[v1] - demands_w[node]
                new_weight_v2 = route_weights[v2] + demands_w[node]

                if new_weight_v1 <= max_capacity_w[v1] and new_weight_v2 <= max_capacity_w[v2]:
                    new_solution = solution.copy()
                    new_solution[v1] = route_v1
                    new_solution[v2] = route_v2
                    if new_solution not in tabu_list:
                        neighbors.append(new_solution)

    # Swap: Swap two customers between two different vehicles
    for v1, v2 in itertools.permutations(vehicles, 2):
        for i in range(1, len(solution[v1]) - 1):
            for j in range(1, len(solution[v2]) - 1):
                node1, node2 = solution[v1][i], solution[v2][j]

                # Modify routes incrementally
                route_v1 = solution[v1][:]
                route_v2 = solution[v2][:]
                route_v1[i], route_v2[j] = node2, node1

                # Incremental weight checks
                new_weight_v1 = route_weights[v1] - demands_w[node1] + demands_w[node2]
                new_weight_v2 = route_weights[v2] - demands_w[node2] + demands_w[node1]

                if new_weight_v1 <= max_capacity_w[v1] and new_weight_v2 <= max_capacity_w[v2]:
                    new_solution = solution.copy()
                    new_solution[v1] = route_v1
                    new_solution[v2] = route_v2
                    if new_solution not in tabu_list:
                        neighbors.append(new_solution)

    # 2-Opt: Reverse a subsequence in a single route
    for v in vehicles:
        route = solution[v]
        for i in range(1, len(route) - 2):  # Exclude depot
            for j in range(i + 1, len(route) - 1):  # Ensure valid subsequence
                new_route = route[:]
                new_route[i:j + 1] = reversed(new_route[i:j + 1])

                # Incremental weight check (no weight change for 2-opt)
                if route_weights[v] <= max_capacity_w[v]:
                    new_solution = solution.copy()
                    new_solution[v] = new_route
                    if new_solution not in tabu_list:
                        neighbors.append(new_solution)

    # Merge routes of two smaller vehicles into a larger vehicle
    for v1, v2, v_large in itertools.permutations(vehicles, 3):
        if len(solution[v1]) > 2 and len(solution[v2]) > 2:
            if max_capacity_w[v_large] >= (max_capacity_w[v1] + max_capacity_w[v2]):
                combined_route = solution[v1][1:-1] + solution[v2][1:-1]  # Exclude depots
                combined_weight = route_weights[v1] + route_weights[v2]

                if combined_weight <= max_capacity_w[v_large]:
                    new_solution = solution.copy()
                    new_solution[v1] = [0, 0]  # Empty route
                    new_solution[v2] = [0, 0]  # Empty route
                    new_solution[v_large] = [0] + combined_route + [0]
                    # Ensure all nodes are visited
                    visited_nodes = {node for route in new_solution.values() for node in route if node != 0}
                    if len(visited_nodes) == 108 and new_solution not in tabu_list:
                        neighbors.append(new_solution)


    # Split a route of a larger vehicle into two smaller vehicles
    for v_large, v1, v2 in itertools.permutations(vehicles, 3):
        if len(solution[v_large]) > 2 and max_capacity_w[v_large] > max_capacity_w[v1] and max_capacity_w[v_large] > max_capacity_w[v2]:
            route_large = solution[v_large][1:-1]  # Exclude depots
            for split_point in range(1, len(route_large)):
                route_v1 = route_large[:split_point]
                route_v2 = route_large[split_point:]

                weight_v1 = sum(demands_w[node] for node in route_v1)
                weight_v2 = sum(demands_w[node] for node in route_v2)

                if weight_v1 <= max_capacity_w[v1] and weight_v2 <= max_capacity_w[v2]:
                    new_solution = solution.copy()
                    new_solution[v_large] = []  # Empty route
                    new_solution[v1] = [0] + route_v1 + [0]
                    new_solution[v2] = [0] + route_v2 + [0]
                    # Ensure all nodes are visited
                    visited_nodes = {node for route in new_solution.values() for node in route if node != 0}
                    if visited_nodes == set(nodes) and new_solution not in tabu_list:
                        neighbors.append(new_solution)
    e_t = time.time()
    run_time = e_t - s_t
    print(f"Time for generating neighbors: {run_time:.4f} seconds")
    return neighbors




def calculate_total_distance(solution, dist_matrix):
    """
    Calculate the total distance for a given solution.
    """
    total_distance = 0
    for route in solution.values():  # Assuming solution is a dictionary
        if len(route) > 2:  # Skip empty or single-point routes
            total_distance += sum(
                dist_matrix[route[i], route[i + 1]] for i in range(len(route) - 1)
            )
    return total_distance

def calculate_total_cost(solution, dist_matrix, Q1, var_cost, fixed_cost):
    # Store the previous solution and costs for incremental updates
    if not hasattr(calculate_total_cost, "previous_solution"):
        calculate_total_cost.previous_solution = {}
        calculate_total_cost.previous_cost = {}
        calculate_total_cost.total_cost = 0

    total_cost = 0

    for vehicle, route in solution.items():
        previous_route = calculate_total_cost.previous_solution.get(vehicle, [])

        # Recalculate cost only if the route has changed
        if route != previous_route:
            if len(route) > 2:  # Skip unused vehicles (routes with only the depot)
                # Fixed cost for the vehicle
                fixed_cost_vehicle = fixed_cost[vehicle]

                # Calculate total distance using a loop
                total_distance = 0
                for i in range(len(route) - 1):
                    total_distance += dist_matrix[route[i], route[i + 1]]

                # Variable cost (distance-based)
                variable_cost_vehicle = total_distance * var_cost[vehicle]

                # Update the previous cost for this vehicle
                calculate_total_cost.previous_cost[vehicle] = fixed_cost_vehicle + variable_cost_vehicle
            else:
                # No cost for unused vehicles
                calculate_total_cost.previous_cost[vehicle] = 0

        # Ensure the vehicle has an entry in previous_cost to avoid KeyError
        if vehicle not in calculate_total_cost.previous_cost:
            calculate_total_cost.previous_cost[vehicle] = 0

        # Add the cost of this vehicle (either recalculated or from previous)
        total_cost += calculate_total_cost.previous_cost[vehicle]

    # Update the previous solution
    calculate_total_cost.previous_solution = solution.copy()

    # Store the total cost
    calculate_total_cost.total_cost = total_cost
    
    return total_cost

In [58]:
def tabu_search(
        nodes, vehicles, dist_matrix, demands_w, max_capacity_w, Q1, var_cost, fixed_cost, max_iter, tabu_tenure,
        time_matrix, start_time, finish_time, time_limit
):
    """
    Tabu Search for minimizing total cost (fixed + variable) in a CVRPTW problem.
    Includes stopping criteria: no improvement for 3 iterations or exceeding the time limit.
    """
    st_time = time.time()

    # Initialize
    current_solution = initialize_solution(nodes, vehicles, dist_matrix, demands_w, max_capacity_w, 
                                           time_matrix, start_time, finish_time)
    best_solution = current_solution
    best_cost = calculate_total_cost(current_solution, dist_matrix, Q1, var_cost, fixed_cost)
    tabu_list = []
    tabu_queue = []
    current_costs = []  # To store the current cost in each iteration
    no_improvement_count = 0  # Counter for iterations without improvement

    for iteration in range(max_iter):
        s_t = time.time()

        # Check time limit
        if time.time() - st_time > time_limit:
            print(f"Stopping early: Exceeded the time limit of {time_limit} seconds.")
            break

        # Generate neighbors
        neighbors = generate_neighbors(
            current_solution, vehicles, nodes, tabu_list, max_capacity_w, demands_w, dist_matrix
        )

        # Evaluate neighbors based on total cost
        best_neighbor = None
        best_neighbor_cost = float("inf")
        for neighbor in neighbors:
            # Validate neighbor feasibility with time window constraints
            feasible = all(
                is_valid_route(neighbor[v], demands_w, max_capacity_w[v], time_matrix, start_time, finish_time,service_time=service_time)
                for v in vehicles
            )

            if feasible:
                neighbor_cost = calculate_total_cost(neighbor, dist_matrix, Q1, var_cost, fixed_cost)
                if neighbor_cost < best_neighbor_cost:
                    best_neighbor = neighbor
                    best_neighbor_cost = neighbor_cost

        # Update current solution if a better neighbor is found
        if best_neighbor and best_neighbor_cost < best_cost:
            current_solution = best_neighbor
            best_cost = best_neighbor_cost
            best_solution = current_solution
            no_improvement_count = 0 
        else:
            no_improvement_count += 1

        # Early stopping condition for no improvement
        if no_improvement_count >= 3:
            print(f"Stopping early: No improvement in the last 3 iterations.")
            break

        # Update tabu list
        tabu_list.append(current_solution)
        if len(tabu_queue) >= tabu_tenure:
            tabu_list.remove(tabu_queue.pop(0))
        tabu_queue.append(current_solution)

        # Record current cost
        current_costs.append(best_cost)

        e_t = time.time()
        run_time = e_t - s_t
        print(f"Iteration {iteration + 1}, Current Cost: {best_cost}, Time for this iteration: {run_time:.2f} seconds")

    en_time = time.time()
    total_time = en_time - st_time
    print(f"Total time in Tabu Search: {total_time:.2f} seconds")

    return best_solution, best_cost, current_costs

In [59]:
print(nodes)
print(vehicles)
print(demands_w)
print(max_capacity_w)   
print(Q1)
print(var_cost)
print(fixed_cost)
print(start_time)
print(finish_time)
print(service_time)
best_solution, best_cost, cost_progress = tabu_search(
    nodes, vehicles, dist_matrix, demands_w, max_capacity_w, Q1=Q1, var_cost=var_cost, fixed_cost=fixed_cost,
    max_iter=100, tabu_tenure=10, time_matrix=time_matrix, start_time=start_time, finish_time=finish_time,time_limit=60
)

print('*'*50)
# Display results
print("Best Solution:")
distance = []
fcost = 0
for v, route in best_solution.items():
    route_distance = sum(dist_matrix[route[i], route[i + 1]] for i in range(len(route) - 1))
    route_time = sum(time_matrix[route[i], route[i + 1]] for i in range(len(route) - 1))
    distance.append(route_distance)
    print(f"Vehicle {v}: Route: {route}, Distance: {route_distance:.2f}, Time: {route_time:.2f}, fixed cost:{fixed_cost[v]}")
    if len(route)>2:    
        fcost += fixed_cost[v]
# print(f"Total Distance: {best_distance}")
print(f"Total cost = {best_cost}")
print(f"Total distance = {sum(distance)}")
print('-'*75)
print(f" Fixed Cost :{fcost}")
print(f" Variable Cost :{best_cost - fcost}")

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
[0, 82, 103, 128, 61, 79, 96, 187, 199, 80, 58, 74, 44, 57, 140, 114, 75, 59, 89, 97, 74, 90, 90, 87, 63, 73, 78, 99, 25, 235, 150, 184, 77, 285, 257, 310, 278, 76]
{0: 2000, 1: 1500, 2: 1200, 3: 1000, 4: 2000, 5: 1500, 6: 1200, 7: 1000, 8: 2000, 9: 1500, 10: 1200, 11: 1000}
[2000, 1500, 1200, 1000, 2000, 1500, 1200, 1000, 2000, 1500, 1200, 1000]
[20, 20, 18, 15, 20, 20, 18, 15, 20, 20, 18, 15]
[3800, 3200, 2500, 2100, 3800, 3200, 2500, 2100, 3800, 3200, 2500, 2100]
[600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600]
[1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380, 1380,